In [ ]:
# EDA for Fault Detection Dataset

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")

BASE_DIR = Path("..").resolve().parent
DATA_DIR = BASE_DIR / "data"
TRAIN_PATH = DATA_DIR / "TRAIN.csv"

TRAIN_PATH, DATA_DIR

In [ ]:
# 1. Load TRAIN.csv and basic inspection

train_df = pd.read_csv(TRAIN_PATH)

print("Shape:", train_df.shape)
display(train_df.head())
print("\nInfo:")
print(train_df.info())
print("\nNull values per column:")
print(train_df.isnull().sum())

In [ ]:
# 2. Class distribution

if "Class" not in train_df.columns:
    raise ValueError("Target column 'Class' not found in TRAIN.csv")

class_counts = train_df["Class"].value_counts().sort_index()

plt.figure(figsize=(5, 4))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values)
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Correlation heatmap

feature_cols = [col for col in train_df.columns if col.startswith("F")]

corr_matrix = train_df[feature_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Feature distributions

n_features = len(feature_cols)
n_cols = 4
n_rows = int(np.ceil(n_features / n_cols))

plt.figure(figsize=(4 * n_cols, 3 * n_rows))

for i, col in enumerate(feature_cols, start=1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(train_df[col].dropna(), bins=30, kde=False)
    plt.title(col)
    plt.xlabel("")
    plt.ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# 5. Constant columns and highly correlated features

# Constant columns
const_cols = [col for col in feature_cols if train_df[col].nunique(dropna=False) <= 1]
print("Constant feature columns (n=%d):" % len(const_cols))
print(const_cols)

# Highly correlated features
threshold = 0.95
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        col_i, col_j = feature_cols[i], feature_cols[j]
        corr_val = corr_matrix.loc[col_i, col_j]
        if abs(corr_val) > threshold:
            high_corr_pairs.append((col_i, col_j, corr_val))

print("\nHighly correlated feature pairs (|corr| > %.2f): n=%d" % (threshold, len(high_corr_pairs)))
for col_i, col_j, corr_val in high_corr_pairs:
    print(f"{col_i} - {col_j}: {corr_val:.3f}")

In [ ]:
# 6. Outlier detection (simple z-score based)

from scipy import stats

z_scores = np.abs(stats.zscore(train_df[feature_cols], nan_policy="omit"))
# Mark samples with any feature having |z| > 3 as potential outliers
outlier_mask = (z_scores > 3).any(axis=1)

n_outliers = outlier_mask.sum()
print(f"Number of potential outlier rows (|z| > 3 in any feature): {n_outliers}")

# Visualize distribution of a few features with and without outliers for reference
example_features = feature_cols[:4]
for col in example_features:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=train_df[col])
    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()